In [1]:
import numpy as np
import pandas as pd

In [2]:
xtraindf = pd.read_csv('mnist_train.csv')
xtestdf = pd.read_csv('mnist_test.csv')
xtrain = xtraindf.to_numpy()
xtest =xtestdf.to_numpy() #first column is label

In [3]:
ytrains = xtrain[:, 0]      
xtrains = xtrain[:, 1:]     

ytests = xtest[:, 0]
xtests= xtest[:, 1:]

xtrain = xtrain / 255.0
xtest = xtest / 255.0

print(f"X train shape: {xtrains.shape}")
print(f"Y train shape: {ytrains.shape}")

X train shape: (59999, 784)
Y train shape: (59999,)


In [4]:
#z=wx+b
#

def relu(z):
    return np.maximum(0,z)

def reluder(z):
    return (z>0).astype(float)

def softmax(Z):
    return np.exp(Z)/np.sum(np.exp(Z), axis=0, keepdims=True)

def one_hot(Y):
    one_hot_Y = np.zeros((Y.size, 10))
    one_hot_Y[np.arange(Y.size), Y.astype(int)] = 1
    return one_hot_Y.T

X_trainfinal = xtrains.T 
Y_trainfinal = one_hot(ytrains)

def cost(expected,predicted):
    cost = -np.mean(expected * np.log(predicted + 1e-8))
    return cost

def cost_mse(expected, predicted):
    return np.mean((predicted - expected) ** 2) / 2

def initparams(layers): #layers is dimensions per layer
    weights = []
    biases = []
    for x in range(1,len(layers)):
        weights.append(np.random.randn(layers[x],layers[x-1])*0.01) #layer one at index 0 and so on
        biases.append(np.random.randn(layers[x],1))
    return [weights , biases]

def forwardpass(weights,biases,input):
    fi = [] #pre-relu
    hi = [] #post-relu
    hi.append(input)
    z=input
    for x in range(0,len(weights)-1):
        f = weights[x].dot(z) + biases[x]
        h = relu(f)
        fi.append(f)
        hi.append(h)
        z=h
    f = weights[-1].dot(z) + biases[-1]
    h = softmax(f)
    fi.append(f)
    hi.append(h)
    return fi,hi


def backprop(fi,hi,weights,biases,label,m):
    fiders = {}
    dz = hi[-1]-label
    fiders[len(fi)-1] = dz
    weightsgrads = [0]*len(weights)
    biasgrads = [0]*len(weights)
    for x in range(len(weights)-2,-1,-1):
        fiders[x] = (weights[x+1].T)@fiders[x+1]*reluder(fi[x])
    for x in range(len(fiders)):
        biasgrads[x] = np.sum(fiders[x], axis=1, keepdims=True)/m
        weightsgrads[x]= fiders[x]@(hi[x].T)/m
    return biasgrads,weightsgrads

def updateparams(weights,biases,learnrate,weightgrads,biasgrads):
    for x in range(len(weights)):
        weights[x] = weights[x] - weightgrads[x]*learnrate
        biases[x] = biases[x] - biasgrads[x]*learnrate
    return weights,biases


In [15]:
#training loop
def trainnn(input,layers,epochs,output,learnrate,batchsize=128):
    weights,biases = initparams(layers)
    for epoch in range(epochs):
        permutation = np.random.permutation(input.shape[1])
        shuffled_X = input[:, permutation]
        shuffled_Y = output[:, permutation]
        for x in range(0,input.shape[1],batchsize):
            Xbatch = shuffled_X[:, x:x + batchsize]
            Ybatch = shuffled_Y[:, x:x + batchsize]

            fi,hi = forwardpass(weights,biases,Xbatch)
            biasgrad,weightgrad = backprop(fi,hi,weights,biases,Ybatch,Xbatch.shape[1])
            prediction = hi[-1]
            updateparams(weights,biases,learnrate,weightgrad,biasgrad)
        if (epoch%10==0):
            fi, hi = forwardpass(weights, biases, input)
            losss = cost(output, hi[-1])
            print(f"epoch {epoch/10}:{losss}")
    
    return weights, biases

def predict(weights, biases, input_data):
    fi, hi = forwardpass(weights, biases, input_data)
    probabilities = hi[-1]
    predictions = np.argmax(probabilities, axis=0)
    return predictions
    

In [16]:
layer_dims = [784, 128, 10] 
weights, biases = trainnn(
    input=X_trainfinal, 
    layers=layer_dims, 
    epochs=50, 
    output=Y_trainfinal, 
    learnrate=0.0001
)

train_predictions = predict(weights, biases, X_trainfinal)
accuracy = np.mean(train_predictions == ytrains) * 100

print(f"Training Accuracy: {accuracy:.2f}%")

epoch 0.0:0.0455342295227385
epoch 1.0:0.019774241711896744
epoch 2.0:0.014218517632450654
epoch 3.0:0.01127221858401091
epoch 4.0:0.009387625046938419
Training Accuracy: 97.70%
